# 01 - Data Collection (SpaceX Falcon 9)
**Goal:** collect Falcon 9 launch data so we can later predict whether the first stage lands successfully.

**Method (simple version):**
1. Call the public **SpaceX REST API (v4)** for past launches.
2. For each launch, look up extra details (rocket, launch site, payload, core/landing info) with more API calls.
3. Keep only **Falcon 9** launches up to 2020-11-13 and single-core, single-payload rows.
4. (Optional) Download the Wikipedia launch tables with web scraping as a second source.
5. Save everything to the `data/` folder.

> Needs internet. Run in Google Colab or Jupyter.

In [ ]:
!pip install -q pandas numpy requests beautifulsoup4 lxml html5lib

In [ ]:
import os, requests, datetime
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
os.makedirs('data', exist_ok=True)

## Step 1 - Request all past launches

In [ ]:
spacex_url = 'https://api.spacexdata.com/v4/launches/past'
response = requests.get(spacex_url, timeout=60)
print('Status code:', response.status_code)   # 200 = success
data = pd.json_normalize(response.json())
print(data.shape)
data.head(3)

## Step 2 - Keep useful columns and simplify
We keep only launches with exactly one core and one payload so each row is one clean record.

In [ ]:
data = data[['rocket','payloads','launchpad','cores','flight_number','date_utc']]
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])
data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)]
print(data.shape)
data.head()

## Step 3 - Helper functions
Each function turns an ID (like a rocket ID) into readable details using another API call. Results are cached so we do not call the API twice for the same ID.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def get_json(url):
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return r.json()

BASE = 'https://api.spacexdata.com/v4/'
BoosterVersion, PayloadMass, Orbit = [], [], []
LaunchSite, Longitude, Latitude = [], [], []
Outcome, Flights, GridFins, Reused, Legs = [], [], [], [], []
LandingPad, Block, ReusedCount, Serial = [], [], [], []

for rid in data['rocket']:
    BoosterVersion.append(get_json(BASE + 'rockets/' + rid)['name'])
for pid in data['payloads']:
    p = get_json(BASE + 'payloads/' + pid)
    PayloadMass.append(p['mass_kg'])
    Orbit.append(p['orbit'])
for lid in data['launchpad']:
    l = get_json(BASE + 'launchpads/' + lid)
    LaunchSite.append(l['name'])
    Longitude.append(l['longitude'])
    Latitude.append(l['latitude'])
for core in data['cores']:
    if core['core'] is not None:
        c = get_json(BASE + 'cores/' + core['core'])
        Block.append(c['block'])
        ReusedCount.append(c['reuse_count'])
        Serial.append(c['serial'])
    else:
        Block.append(None); ReusedCount.append(None); Serial.append(None)
    Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
    Flights.append(core['flight'])
    GridFins.append(core['gridfins'])
    Reused.append(core['reused'])
    Legs.append(core['legs'])
    LandingPad.append(core['landpad'])
print('Collected all details')

## Step 4 - Build the dataset and keep Falcon 9 only

In [ ]:
launch_dict = {
    'FlightNumber': list(data['flight_number']),
    'Date': list(data['date']),
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude,
}
df = pd.DataFrame(launch_dict)
data_falcon9 = df[df['BoosterVersion'] == 'Falcon 9'].copy()
data_falcon9['FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
print('Falcon 9 launches:', data_falcon9.shape[0])
data_falcon9.head()

In [ ]:
data_falcon9.to_csv('data/spacex_api_raw.csv', index=False)
print('Saved data/spacex_api_raw.csv')
print(data_falcon9.isnull().sum())

## Step 5 (Optional but recommended) - Web scraping from Wikipedia
This gives a second data source and shows web scraping skill. We use a **fixed archived version** of the page so results do not change over time.

In [ ]:
from bs4 import BeautifulSoup
wiki_url = ('https://en.wikipedia.org/w/index.php?'
            'title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922')
headers = {'User-Agent': 'Mozilla/5.0 (student capstone project)'}
html = requests.get(wiki_url, headers=headers, timeout=60).text
soup = BeautifulSoup(html, 'html.parser')
print(soup.title.string)

In [ ]:
tables = soup.find_all('table', 'wikitable plainrowheaders collapsible')
print('Launch tables found:', len(tables))

rows = []
for t in tables:
    for tr in t.find_all('tr'):
        th = tr.find('th')
        # a launch row starts with a flight number in the header cell
        if th and th.get_text(strip=True).isdigit():
            cells = [td.get_text(' ', strip=True) for td in tr.find_all('td')]
            rows.append([th.get_text(strip=True)] + cells)

wiki_df = pd.DataFrame(rows)
print(wiki_df.shape)
wiki_df.head()

In [ ]:
wiki_df.to_csv('data/spacex_wiki_raw.csv', index=False)
print('Saved data/spacex_wiki_raw.csv')

## Summary
- Data source 1: SpaceX REST API v4 -> `data/spacex_api_raw.csv`
- Data source 2: Wikipedia launch tables (scraped) -> `data/spacex_wiki_raw.csv`
- Next notebook: **02_data_wrangling.ipynb** uses the API file.